In [ ]:
import os
import numpy as np
os.environ.pop("DEBUG", None)
# os.environ["DEBUG"] = "2"
from tinygrad import Tensor, Device, TinyJit
from tinygrad.nn.optim import AdamW
from tinygrad.nn.state import get_parameters
from game import GameBatch, GameResult, Color, MAX_MOVES
from model import Model, Config, init_weights
from helpers._evaluator_old import Evaluator

print("backend:", Device.DEFAULT)   # want METAL on your Mac

# batch of games = how many can be run at once on the model
B = 64
rng = np.random.default_rng(42)

backend: METAL


In [2]:
batch = GameBatch(B)
config = Config()
model = Model(config)
init_weights(model, config)
evaluator = Evaluator(model, B)

class MCTSNode:
	__slots__ = ("n_moves", "terminal", "term_value", "expanded", "P", "N", "W", "children")

	def __init__(self, n_moves, terminal=False, term_value=0.0):
		self.n_moves = n_moves
		self.terminal = terminal
		self.term_value = term_value
		self.expanded = False
		self.P = self.N = self.W = self.children = None
	
	def expand(self, priors):
		self.P = priors
		self.N = np.zeros(self.n_moves, dtype=np.float32)
		self.W = np.zeros(self.n_moves, dtype=np.float32)
		self.children = [None] * self.n_moves
		self.expanded = True

def _terminal_value(result): return -1 if result == GameResult.CHECKMATE else 0.0
def _select(node: MCTSNode, c_puct: int):
	N, W, P = node.N, node.W, node.P
	N_total = N.sum()
	Q = np.where(N > 0, W / np.maximum(N, 1.0), 0.0)
	U = Q + c_puct * P * np.sqrt(N_total + 1.0) / (1.0 + N)
	return int(U.argmax())

# run mcts with n_sims simulations on all games in a GameBatch
def run_mcts(
	batch: GameBatch, 
	evaluator: Evaluator, 
	rng: np.random.Generator,
	n_sims: int = 50,
	c_puct: int = 1.5,
	temperature: int = 1.0,
	dirichlet_alpha = 0.3,
):
	B = len(batch)
	games = [batch[i] for i in range(B)]
	active = batch.active
	state = tuple(a.copy() for a in batch.get_encoding())

	priors_root, _ = evaluator.eval(batch.get_encoding())
	n_moves = batch.num_moves_all()
	roots: list[MCTSNode] = []

	# expand roots
	for i in range(B):
		if not active[i]:
			roots.append(None)
			continue
		n = int(n_moves[i])
		node = MCTSNode(n)
		P = priors_root[i, :n].astype(np.float32).copy()
		if dirichlet_alpha and n > 0:
			noise = rng.dirichlet([dirichlet_alpha] * n).astype(np.float32)
			P = 0.75 * P + 0.25 * noise
		node.expand(P)
		roots.append(node)
	
	for _sim in range(n_sims):
		leaves: list[tuple[int, MCTSNode, any]] = []
		for i in range(B):
			if not active[i]: continue
			game, node, path = games[i], roots[i], []
			while node.expanded:
				a = _select(node, c_puct)
				path.append((node, a))
				game.play(a)
				child = node.children[a]
				if child is None:
					result = game.result()
					term = result != GameResult.ONGOING
					child = MCTSNode(game.num_moves(), term, 
						_terminal_value(result) if term else 0.0
					)
					node.children[a] = child
				node = child
			leaves.append((i, node, path))
		priors_b, values_b = evaluator.eval(batch.get_encoding())

		for i, leaf, path in leaves:
			if leaf.terminal:
				v = leaf.term_value
			else:
				leaf.expand(priors_b[i, :leaf.n_moves].astype(np.float32).copy())
				v = float(values_b[i])
			for node, a in reversed(path):
				v = -v
				node.N[a] += 1.0
				node.W[a] += v
			for _ in range(len(path)):
				games[i].undo()
	
	pi = np.zeros((B, MAX_MOVES), dtype=np.float32)
	moves = np.zeros(B, dtype=np.int32)
	for i in range(B):
		if not active[i]: continue
		N, n = roots[i].N, roots[i].n_moves
		total = N.sum()
		if total > 0:
			pi[i, :n] = N / total
		if temperature <= 1e-3 or total == 0:
			moves[i] = int(N.argmax()) if total > 0 else 0
		else:
			heated = pow(N, 1.0 / temperature)
			moves[i] = int(rng.choice(n, p=heated / heated.sum()))
	return state, pi, moves

class ReplayBuffer:
	def __init__(self, capacity: int):
		C = capacity
		self.cap, self.size, self.ptr = C, 0, 0
		self.board = np.zeros((C, 64), np.uint8)
		self.castling = np.zeros((C,), np.uint8)
		self.ep = np.zeros((C,), np.int8)
		self.rep = np.zeros((C,), np.uint8)
		self.clock = np.zeros((C,), np.uint8)
		self.moves = np.zeros((C, MAX_MOVES, 4), np.uint8)
		self.num_moves = np.zeros((C,), np.int32)
		self.pi = np.zeros((C, MAX_MOVES), np.float32)
		self.z = np.zeros((C,), np.float32)
	
	def add(self, enc, pi, z):
		i = self.ptr
		b, c, e, r, cl, m, nm = enc
		self.board[i] = b
		self.castling[i] = c
		self.ep[i] = e
		self.rep[i] = r
		self.clock[i] = cl
		self.moves[i] = m
		self.num_moves[i] = nm
		self.pi[i] = pi
		self.z[i] = z
		self.ptr = (self.ptr + 1) % self.cap
		self.size = min(self.size + 1, self.cap)
	
	def sample(self, mb: int, rng: np.random.Generator):
		idx = rng.integers(0, self.size, size=mb)
		return (
			self.board[idx], 
			self.castling[idx], 
			self.ep[idx], 
			self.rep[idx],
			self.clock[idx], 
			self.moves[idx], 
			self.num_moves[idx],
			self.pi[idx], 
			self.z[idx]
		)

In [13]:
class ReplayBuffer:
	def __init__(self, capacity: int):
		C = capacity
		self.cap, self.size, self.ptr = C, 0, 0
		self.board = np.zeros((C, 64), np.uint8)
		self.castling = np.zeros((C,), np.uint8)
		self.ep = np.zeros((C,), np.int8)
		self.rep = np.zeros((C,), np.uint8)
		self.clock = np.zeros((C,), np.uint8)
		self.moves = np.zeros((C, MAX_MOVES, 4), np.uint8)
		self.num_moves = np.zeros((C,), np.int32)
		self.pi = np.zeros((C, MAX_MOVES), np.float32)
		self.z = np.zeros((C,), np.float32)
	
	def add(self, enc, pi, z):
		i = self.ptr
		b, c, e, r, cl, m, nm = enc
		self.board[i] = b
		self.castling[i] = c
		self.ep[i] = e
		self.rep[i] = r
		self.clock[i] = cl
		self.moves[i] = m
		self.num_moves[i] = nm
		self.pi[i] = pi
		self.z[i] = z
		self.ptr = (self.ptr + 1) % self.cap
		self.size = min(self.size + 1, self.cap)
	
	def sample(self, mb: int, rng: np.random.Generator):
		idx = rng.integers(0, self.size, size=mb)
		return (
			self.board[idx], 
			self.castling[idx], 
			self.ep[idx], 
			self.rep[idx],
			self.clock[idx], 
			self.moves[idx], 
			self.num_moves[idx],
			self.pi[idx], 
			self.z[idx]
		)

def save(self, path):
	s = self.size                          # only the filled entries
	np.savez_compressed(
		path,
		board=self.board[:s],
		castling=self.castling[:s],
		ep=self.ep[:s],
		rep=self.rep[:s],
		clock=self.clock[:s],
		moves=self.moves[:s],
		num_moves=self.num_moves[:s],
		pi=self.pi[:s],
		z=self.z[:s],
		meta=np.array([self.cap, self.size, self.ptr], dtype=np.int64),
	)

@classmethod
def load(cls, path):
	with np.load(path) as d:
		cap, size, ptr = (int(x) for x in d["meta"])
		buf = cls(cap)                     # allocates the full zeroed arrays
		s = size
		buf.board[:s] = d["board"]
		buf.castling[:s] = d["castling"]
		buf.ep[:s] = d["ep"]
		buf.rep[:s] = d["rep"]
		buf.clock[:s] = d["clock"]
		buf.moves[:s] = d["moves"]
		buf.num_moves[:s] = d["num_moves"]
		buf.pi[:s] = d["pi"]
		buf.z[:s] = d["z"]
		buf.size, buf.ptr = size, ptr
	return buf

In [ ]:
# exclude = {id(model.q_head.weight), id(model.q_head.bias)}
# params = [t for t in get_parameters(model) if id(t) not in exclude]
opt = AdamW(get_parameters(model), lr=1e-3)
step = 0
MB = 256
@TinyJit
def train_step(board, castling, ep, rep, clock, moves, num_moves, pi, z):
	p, v = model(board, castling, ep, rep, clock, moves, num_moves)
	policy_loss = -(pi * p.log_softmax(axis=-1)).sum(axis=-1).mean()
	value_loss = (pow(v.squeeze(-1) - z, 2)).mean()
	loss = policy_loss + value_loss
	opt.zero_grad()
	loss.backward()
	opt.step()
	return loss.realize()

In [ ]:
buffer = ReplayBuffer(200_000)
traj   = [[] for _ in range(B)]        # per-game (encoding, pi, side) until the game ends
MIN_BUFFER = 4_000                     # warm up before training
temp = 1.0

In [7]:
while step < 50_000:
	# ---- one self-play ply across all B games ----
	states, pi, mv = run_mcts(batch, evaluator, rng=rng, n_sims=50, temperature=temp)
	side = batch.to_moves()
	for i in range(B):                 # record the position we just searched
		enc_i = tuple(s[i].copy() for s in states)
		traj[i].append((enc_i, pi[i].copy(), int(side[i])))

	results = batch.play_batch(mv)
	loser   = batch.to_moves()         # side to move AFTER the move = mated side on checkmate

	for i in range(B):                 # label + flush + restart finished games
		if results[i] == GameResult.ONGOING:
			continue
		if results[i] == GameResult.CHECKMATE:
			winner = 1 - int(loser[i])
			zs = [1.0 if s == winner else -1.0 for (_, _, s) in traj[i]]
		else:                          # stalemate / 50-move / repetition
			zs = [0.0] * len(traj[i])
		for (enc, p_, _), zz in zip(traj[i], zs):
			buffer.add(enc, p_, zz)
		traj[i] = []
		batch[i].reset()

	# ---- one training step once warmed up ----
	if buffer.size >= MIN_BUFFER:
		Tensor.training = True
		loss = train_step(*[Tensor(a) for a in buffer.sample(MB, rng)])
		if step % 100 == 0:
			print(f"step {step}  buffer {buffer.size}  loss {loss.item():.3f}")
	step += 1

step 700  buffer 34042  loss 2.901
step 800  buffer 38392  loss 2.908
step 900  buffer 46528  loss 2.904
step 1000  buffer 55566  loss 2.875


KeyboardInterrupt: 

In [8]:
def random_moves(batch, active, rng):
    nm = batch.num_moves_all()
    mv = np.zeros(len(batch), np.int32)
    for i in range(len(batch)):
        if active[i] and nm[i] > 0:
            mv[i] = rng.integers(0, int(nm[i]))
    return mv

def eval_vs_random(evaluator, rng, n_games=32, max_plies=250, n_sims=25,
                   model_color=Color.WHITE):
    eb = GameBatch(n_games)                      # fresh batch, doesn't touch training state
    for t in range(max_plies):
        active = eb.active
        if not active.any():
            break
        if (t & 1) == int(model_color):          # model's turn
            _, _, mv = run_mcts(eb, evaluator, rng=rng, n_sims=n_sims,
                                temperature=0.0, dirichlet_alpha=0.0)
        else:                                    # random bot's turn
            mv = random_moves(eb, active, rng)
        eb.play_batch(mv, mask=active)

    results, to_move = eb.results(), eb.to_moves()
    w = l = d = u = 0
    for i in range(n_games):
        r = results[i]
        if r == GameResult.ONGOING:
            u += 1                               # hit the ply cap, never finished
        elif r == GameResult.CHECKMATE:
            # side to move at the terminal position is the mated side (the loser)
            if int(to_move[i]) != int(model_color):
                w += 1
            else:
                l += 1
        else:
            d += 1                               # stalemate / 50-move / repetition
    return w, l, d, u

w, l, d, u = eval_vs_random(evaluator, rng, n_games=32, n_sims=25)
n = 32
print(f"model vs random  ({n} games, model as White, 250-ply cap)")
print(f"  W {w}   L {l}   D {d}   unfinished {u}")
print(f"  win rate {w/n:.0%}    score {(w + 0.5*(d+u))/n:.2f}")

model vs random  (32 games, model as White, 250-ply cap)
  W 11   L 0   D 8   unfinished 13
  win rate 34%    score 0.67


In [9]:
from tinygrad.nn.state import get_state_dict, safe_save, safe_load, load_state_dict

# save
safe_save(get_state_dict(model), "chess_model.safetensors")

In [ ]:
buffer.save("replay_buffer.npz")
buffer = ReplayBuffer.load("replay_buffer.npz")